# Building a BPE tokenizer

This notebook builds a byte-level BPE tokenizer step by step. It follows
Andrej Karpathy's video [Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE).
Each part writes the idea in a few lines first, then uses the matching code from `src/bytepair`.

1. Text to bytes
2. Byte pair encoding
3. Encoding and decoding
4. Regex splitting
5. Special tokens
6. Training on a book
7. Matching GPT-4

## 1. Text to bytes

A language model reads a sequence of integers. We need a fixed way to turn text into integers and back.

Python strings are sequences of Unicode code points. There are about 150,000 of them, and the set keeps growing,
so using code points as tokens gives a large, unstable vocabulary.

UTF-8 encodes each code point as 1 to 4 bytes. Bytes give a fixed base vocabulary of 256 values that can represent
any text. The cost is length: non-ASCII text turns into many tokens.

In [1]:
text = "hi 안녕 😉"
print([ord(c) for c in text])
print(list(text.encode("utf-8")))
print(len(text), "code points ->", len(text.encode("utf-8")), "bytes")

[104, 105, 32, 50504, 45397, 32, 128521]
[104, 105, 32, 236, 149, 136, 235, 133, 149, 32, 240, 159, 152, 137]
7 code points -> 14 bytes


Raw bytes make sequences long, and a transformer's attention cost grows with sequence length.
BPE shortens the sequence by adding new tokens for byte sequences that appear often.

## 2. Byte pair encoding

The training loop:

1. Count every adjacent pair of tokens.
2. Pick the most frequent pair.
3. Replace it everywhere with a new token id (256, 257, ...).
4. Repeat until the vocabulary is the size you want.

The two helpers are short.

In [2]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

ids = list("aaabdaaabac".encode("utf-8"))
print(get_stats(ids))

{(97, 97): 4, (97, 98): 2, (98, 100): 1, (100, 97): 1, (98, 97): 1, (97, 99): 1}


The Wikipedia example `aaabdaaabac` becomes `XdXac` after three merges:

In [3]:
merges = {}
for i in range(3):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    ids = merge(ids, pair, 256 + i)
    merges[pair] = 256 + i
    print(f"merge {pair} -> {256 + i}: {ids}")

merge (97, 97) -> 256: [256, 97, 98, 100, 256, 97, 98, 97, 99]
merge (256, 97) -> 257: [257, 98, 100, 257, 98, 97, 99]
merge (257, 98) -> 258: [258, 100, 258, 97, 99]


`merges` is the trained tokenizer. The library version is `BasicTokenizer` in `src/bytepair/basic.py`:

In [4]:
from bytepair.basic import BasicTokenizer

tok = BasicTokenizer()
tok.train("aaabdaaabac", 256 + 3)
tok.merges

{(97, 97): 256, (256, 97): 257, (257, 98): 258}

## 3. Encoding and decoding

Decoding concatenates each token's bytes and decodes UTF-8. A token sequence can end inside a multi-byte
character, so we decode with `errors="replace"` instead of raising.

Encoding applies merges to new text in the order they were learned. At each step it merges the pair
with the lowest merge id that is present. Order matters because later merges are built on earlier ones.

In [5]:
corpus = "the cat sat on the mat. the cat ate the rat. " * 20
tok = BasicTokenizer()
tok.train(corpus, 280)

ids = tok.encode("the cat sat")
print(ids)
print([tok.vocab[i] for i in ids])
print(tok.decode(ids))

[259, 262, 115, 256]
[b'the ', b'cat ', b's', b'at']
the cat sat


`BasicTokenizer` merges across word and punctuation boundaries. On this repetitive corpus it learns long
tokens that mix words, punctuation and spaces, such as `b'at. the '`:

In [6]:
[tok.vocab[i] for i in range(256, 280)]

[b'at',
 b'e ',
 b'th',
 b'the ',
 b'at ',
 b' the ',
 b'cat ',
 b'at.',
 b'at. the ',
 b'at. the cat ',
 b'sat ',
 b'sat o',
 b'sat on',
 b'sat on the ',
 b'sat on the m',
 b'sat on the mat. the cat ',
 b'sat on the mat. the cat at',
 b'sat on the mat. the cat ate ',
 b'sat on the mat. the cat ate the ',
 b'sat on the mat. the cat ate the r',
 b'sat on the mat. the cat ate the rat. the cat ',
 b'sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat ',
 b'sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat ',
 b'sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat on the mat. the cat ate the rat. the cat sat

## 4. Regex splitting

GPT-2 and later models split the text into chunks with a regex before running BPE. Merges happen only inside a chunk.
This keeps letters, numbers, punctuation and whitespace apart, so `dog`, `dog.` and `dog!` share the token `dog`.

The GPT-4 pattern also:
- matches contractions case-insensitively (`'s`, `'LL`),
- splits numbers into groups of at most three digits,
- attaches a leading space to the following word.

In [7]:
from bytepair.regex import GPT4_SPLIT_PATTERN, RegexTokenizer

tok = RegexTokenizer()
tok.compiled_pattern.findall("Hello world's 1234567 dogs!!\n\n    def f(x):")

['Hello',
 ' world',
 "'s",
 ' ',
 '123',
 '456',
 '7',
 ' dogs',
 '!!\n\n',
 '   ',
 ' def',
 ' f',
 '(x',
 '):']

Training changes in one way: count pairs inside each chunk and never across chunks.

Text repeats chunks heavily (`" the"` appears thousands of times in a book). `RegexTokenizer.train`
counts unique chunks, trains on each unique chunk once, and weights its pair counts by frequency.
This gives the same merges as processing every chunk and is much faster (about 15 seconds for a 1,024-token
vocab on a 700 KB book). The same idea makes encoding fast: each unique chunk is encoded once and cached.

## 5. Special tokens

Special tokens such as `<|endoftext|>` mark structure, like document boundaries or chat turns. BPE never
produces them. The encoder must find them in the text and insert their ids directly.

Text from users can contain the string `<|endoftext|>`. Encoding it as the special token would let users inject
structure, so `encode` raises by default. You must allow special tokens explicitly, as in tiktoken.

In [8]:
tok = RegexTokenizer()
tok.train(corpus, 280)
tok.register_special_tokens({"<|endoftext|>": 280})

text = "the cat<|endoftext|>the rat"
print("all: ", tok.encode(text, allowed_special="all"))
print("none:", tok.encode(text, allowed_special="none"))
try:
    tok.encode(text)
except ValueError as e:
    print("default:", e)

all:  [258, 261, 280, 258, 271]
none: [258, 261, 60, 124, 101, 110, 100, 111, 102, 116, 101, 120, 116, 124, 62, 258, 271]
default: text contains special token '<|endoftext|>'


## 6. Training on a book

`scripts/train_showcase.py` trains a 1,024-token `RegexTokenizer` on *Pride and Prejudice* and saves it to
`models/pride.model`. Here we load it and look at what it learned.

In [9]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pride = RegexTokenizer()
pride.load(ROOT / "models" / "pride.model")

print("first merges:", [pride.vocab[i] for i in range(256, 276)])
print("last merges: ", [pride.vocab[i] for i in range(1004, 1024)])

first merges: [b'he', b' t', b' a', b'in', b' w', b' s', b're', b' o', b'ha', b'  ', b'er', b'ou', b' the', b'en', b'on', b'is', b' b', b'it', b' m', b'ed']
last merges:  [b' come', b' aunt', b' ill', b' gi', b'ret', b' inst', b'ember', b'erm', b'iam', b' does', b' gener', b'shire', b' word', b'ack', b' form', b' where', b' R', b' dist', b' bec', b' hand']


The first merges are common English fragments (`he`, ` t`, ` the`). Character names get their own tokens early,
because they appear so often in this book:

In [10]:
names = [b"Elizabeth", b" Darcy", b" Bennet", b" Bingley", b" Wickham"]
{n.decode(): next(i for i, t in pride.vocab.items() if t == n) for n in names}

{'Elizabeth': 400,
 ' Darcy': 472,
 ' Bennet': 531,
 ' Bingley': 559,
 ' Wickham': 697}

Compression measures how many bytes one token covers on average. More bytes per token means shorter sequences.
A tokenizer does best on text like its training data.

In [11]:
import tiktoken

cl100k = tiktoken.get_encoding("cl100k_base")
samples = {
    "Austen": "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.",
    "Python": "def fib(n):\n    if n < 2:\n        return n\n    return fib(n - 1) + fib(n - 2)\n",
    "Korean": "안녕하세요, 만나서 반갑습니다.",
}
for name, s in samples.items():
    n = len(s.encode("utf-8"))
    print(f"{name:7} pride: {n / len(pride.encode(s)):.2f} bytes/token   cl100k: {n / len(cl100k.encode(s)):.2f}")

Austen  pride: 2.54 bytes/token   cl100k: 4.50
Python  pride: 1.50 bytes/token   cl100k: 2.60
Korean  pride: 1.00 bytes/token   cl100k: 3.07


On Austen, the 1k-vocab tokenizer reaches a bit over half of GPT-4's compression with 1% of the vocabulary.
On code and Korean it drops to close to one byte per token, because it never saw that kind of text in training.
Training data decides what a tokenizer is good at.

## 7. Matching GPT-4

tiktoken publishes the cl100k_base vocabulary as `{token bytes: rank}`, not as a merge list. `GPT4Tokenizer`
rebuilds the merges: for each token, run BPE on its bytes using only merges with a lower rank. The two parts
left over are the pair that merges into it.

One detail: in tiktoken the 256 single bytes have ranks that differ from their byte values. `GPT4Tokenizer`
stores this as a byte shuffle and applies it before BPE and reverses it after decoding.

With that, the output matches tiktoken exactly:

In [12]:
from bytepair.gpt4 import GPT4Tokenizer

gpt4 = GPT4Tokenizer()
text = (ROOT / "data" / "pride_and_prejudice.txt").read_text() if (ROOT / "data" / "pride_and_prejudice.txt").exists() else samples["Austen"]
ours = gpt4.encode(text)
print(len(ours), "tokens, identical to tiktoken:", ours == cl100k.encode(text))

171510 tokens, identical to tiktoken: True


## What tokenization explains

- LLMs are weaker on non-English text partly because it costs more tokens for the same content.
- Arithmetic is hard when numbers split into arbitrary chunks (`1234567` becomes `123`, `456`, `7`).
- Trailing spaces and odd whitespace change tokenization and can hurt completions.
- Special tokens must be handled explicitly, or user text can inject them.